# Income & inequality vs the birth rate — WangGaoyang

AN6100 group project · Part 4 · Presenter: Wang Gaoyang

**Question.** Did household income or inequality move with Singapore's total fertility rate (TFR), 2010–2025?

**Data** (all SingStat Table Builder; every raw value is in `Income_WangGaoyang_Data.csv`, `part = raw`; the original downloads are in `raw/`):

| Table | What | Used for |
|---|---|---|
| 17872 | Average and median monthly household employment income per household member (incl. employer CPF), resident employed households, S$ | Charts 1, 5, 6 |
| M213751 | CPI All Items, monthly, 2024 = 100 | Chart 1 (real income) |
| 17885 | Real change in average income per member vs previous year, all households and deciles D1–D9, % | Chart 2 |
| 17892 | Gini coefficient before and after government transfers & taxes | Charts 3, 4, 6 |
| M810361 | Real change in average income per member, lowest 10% and top 10%, % | Chart 4 |
| 17883 | Average income per member by decile (only to check its total equals 17872) | check |
| M810091 | Total fertility rate | all charts |

**Method.** Real income = nominal ÷ CPI × 100, with CPI All Items expressed as 2010 = 100 (SingStat's method; it reproduces 17885 exactly).
Pearson r and two-sided p-values (`scipy.stats.pearsonr`):
- *Levels*: r of x[t] and TFR[t]
- *Yearly change*: r of Δx[t] = x[t] − x[t−1] and ΔTFR[t]
- *Yearly change, lag L*: r of Δx[t] and ΔTFR[t+L] (births respond to decisions about a year later)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import pearsonr

pio.renderers.default = "plotly_mimetype+notebook_connected"
Y0, Y1 = 2010, 2025

HERE = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "members" / "4_Income_WangGaoyang"]
            if (p / "Income_WangGaoyang_Data.csv").is_file())
data = pd.read_csv(HERE / "Income_WangGaoyang_Data.csv")
raw, clean_saved = data[data.part == "raw"], data[data.part == "clean"]
print(raw.groupby("table").size().rename("raw values").to_string())

table
17872      104
17883      260
17885      240
17892       52
M213751    204
M810091     66
M810361     50


## Step 1 · Clean and merge

Each raw table is turned into one column per series and one row per year, then joined on `Year` (2010–2025).
CPI: annual value = mean of the 12 months; rebased so 2010 = 100 (CPI_t ÷ CPI_2010 × 100). Changing the base year changes no growth rate.

In [2]:
def series(table, name):
    s = raw[(raw.table == table) & (raw.series == name)]
    return pd.Series(s.value.values, index=s.period.astype(int).values, name=name).sort_index()

nominal_avg = series("17872", "Average, resident employed households")
nominal_med = series("17872", "Median, resident employed households")
assert (series("17883", "Total") == nominal_avg).all()          # 17883 total = 17872 average (same series)

cpi_m = raw[raw.table == "M213751"].copy()
cpi_m["Year"] = cpi_m.period.str[:4].astype(int)
cpi_2024 = cpi_m.groupby("Year").value.mean()                    # annual mean, 2024 = 100 (as published)
cpi_2010 = cpi_2024 / cpi_2024.loc[Y0] * 100                     # 2010 = 100

DEC = ["Total", "1st (Lowest)", "2nd", "3rd", "4th", "5th", "6th", "7th", "8th", "9th"]
clean = pd.DataFrame(index=pd.Index(range(Y0, Y1 + 1), name="Year"))
clean["TFR"] = series("M810091", "Total Fertility Rate (TFR)")
clean["PM_Avg"], clean["PM_Median"] = nominal_avg, nominal_med
clean["CPI_All_Items_2024_100"], clean["CPI_All_Items_2010_100"] = cpi_2024.round(3), cpi_2010.round(3)
clean["PM_Avg_Real_2010"] = (nominal_avg / cpi_2010 * 100).round(2)
clean["PM_Median_Real_2010"] = (nominal_med / cpi_2010 * 100).round(2)
clean["Real_Growth_%"] = series("17885", "Total")
for i, d in enumerate(DEC[1:], 1):
    clean[f"Real_Growth_D{i}"] = series("17885", d)
clean["Gini_Before_Transfers"] = series("17892", "Gini before government transfers & taxes")
clean["Gini_After_Transfers"] = series("17892", "Gini after government transfers & taxes")
clean["Real_Growth_D1_M810361"] = series("M810361", "Real change in average income per member, 1st Decile (Lowest)")
clean["Real_Growth_D10_M810361"] = series("M810361", "Real change in average income per member, 10th Decile (Highest)")

# check 1: the real % change computed this way equals 17885's published total in every year
diff = ((clean.PM_Avg_Real_2010.pct_change() * 100).round(1) - clean["Real_Growth_%"]).loc[2011:2024].abs().max()
assert diff == 0, diff
# check 2: identical to the cleaned table saved in the CSV (part = clean)
saved = clean_saved.assign(Year=clean_saved.period.astype(int)).pivot(index="Year", columns="series", values="value")
pd.testing.assert_frame_equal(clean[saved.columns].dropna(how="all"), saved.reindex(clean.index)[saved.columns].dropna(how="all"),
                              check_names=False, check_dtype=False)
print("real income reproduces 17885: max difference", diff, "· clean table matches the CSV")
clean.round(3)

real income reproduces 17885: max difference 0.0 · clean table matches the CSV


,TFR,PM_Avg,PM_Median,CPI_All_Items_2024_100,CPI_All_Items_2010_100,PM_Avg_Real_2010,PM_Median_Real_2010,Real_Growth_%,Real_Growth_D1,Real_Growth_D2,...,Real_Growth_D4,Real_Growth_D5,Real_Growth_D6,Real_Growth_D7,Real_Growth_D8,Real_Growth_D9,Gini_Before_Transfers,Gini_After_Transfers,Real_Growth_D1_M810361,Real_Growth_D10_M810361
Year,,,,,,,,,,,,,,,,,,,,,
2010,1.15,2711.0,1850.0,75.099,100.000,2711.00,1850.00,4.4,4.2,5.6,...,5.1,3.8,3.5,3.3,3.4,4.0,0.472,0.425,4.2,4.8
2011,1.20,2927.0,1997.0,79.043,105.251,2780.97,1897.37,2.6,5.8,4.0,...,2.6,2.4,2.4,2.2,1.8,1.0,0.473,0.423,5.8,3.2
2012,1.29,3145.0,2129.0,82.664,110.073,2857.21,1934.18,2.7,-1.2,1.0,...,2.3,2.1,1.9,1.4,1.1,1.6,0.478,0.432,-1.2,5.1
2013,1.19,3207.0,2250.0,84.618,112.675,2846.24,1996.89,-0.4,2.4,1.9,...,3.1,3.6,3.0,2.6,2.4,1.5,0.463,0.409,2.4,-5.2
2014,1.25,3421.0,2383.0,85.492,113.839,3005.12,2093.30,5.6,5.1,6.7,...,5.8,4.8,4.6,4.6,4.6,5.6,0.464,0.411,5.1,6.6
2015,1.24,3629.0,2503.0,85.046,113.245,3204.56,2210.25,6.6,10.7,8.3,...,6.7,5.9,6.2,6.3,5.7,5.8,0.463,0.409,10.7,7.3
2016,1.20,3694.0,2591.0,84.596,112.645,3279.33,2300.15,2.3,1.4,3.4,...,2.4,3.3,3.5,3.8,4.3,3.2,0.458,0.401,1.4,0.3
2017,1.16,3843.0,2706.0,85.084,113.295,3392.04,2388.46,3.4,2.1,2.8,...,3.0,3.6,4.0,3.7,3.8,4.5,0.459,0.402,2.1,2.6
2018,1.14,3977.0,2799.0,85.457,113.792,3494.98,2459.75,3.0,2.7,2.6,...,3.4,3.2,2.5,3.2,4.0,4.0,0.458,0.403,2.7,2.3


## Step 2 · Statistics helpers and chart style (same as the dashboard)

In [3]:
BLUE, ORANGE, AQUA, VIOLET, GOLD, RED = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7", "#c98a00", "#e34948"
MUTED, GRID, LIGHT = "#6b6a66", "#ececea", "#d4d3cd"
DIVERGING = [[0, BLUE], [0.5, "#efeeea"], [1, RED]]
df = clean.reset_index()
T = clean.TFR


def pair(x, mode="Levels", lag=0):
    """x[t] against TFR[t+lag]; for 'Yearly change' both are differenced (this year minus last year)."""
    xs = clean[x].dropna().copy(); xs.index = xs.index + lag
    d = pd.concat([xs.rename("x"), T.rename("y")], axis=1).dropna()
    if mode == "Yearly change":
        d = d.reindex(range(d.index.min(), d.index.max() + 1)).diff().dropna()
    return d


def r_of(x, mode="Levels", lag=0):
    d = pair(x, mode, lag)
    r, p = pearsonr(d.x, d.y)
    return r, p, len(d)


def p_text(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}" if p < 0.01 else f"{p:.2f}"


def style(fig, h=380):
    fig.update_layout(height=h, template="plotly_white", margin=dict(l=10, r=10, t=60, b=10), font=dict(size=12),
                      legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0, title=None), hovermode="x unified")
    fig.update_xaxes(showgrid=False); fig.update_yaxes(gridcolor=GRID, zeroline=False)
    return fig


def add_tfr_axis(fig):
    fig.add_trace(go.Scatter(x=df.Year, y=df.TFR, name="TFR (right axis)", yaxis="y2", mode="lines+markers",
                             line=dict(color=BLUE, width=2.8, dash="dot"), marker=dict(size=5, symbol="diamond"),
                             hovertemplate="TFR %{y:.2f}<extra></extra>"))
    lo, hi = df.TFR.min(), df.TFR.max(); pad = (hi - lo) * 0.15
    fig.update_layout(yaxis2=dict(title=dict(text="TFR, children per woman", font=dict(color=BLUE)), overlaying="y",
                                  side="right", showgrid=False, tickformat=".2f", tickfont=dict(color=BLUE),
                                  range=[lo - pad, hi + pad]))
    return fig

## Chart 1 · Income per household member: nominal and real

Source: SingStat 17872, M213751, M810091 · Real = nominal ÷ CPI (2010 = 100) × 100.

In [4]:
fig = go.Figure()
for c, n, col, dash in [("PM_Median", "Median, nominal", ORANGE, "solid"), ("PM_Median_Real_2010", "Median, real (2010 S$)", AQUA, "solid"),
                        ("PM_Avg", "Average, nominal", ORANGE, "dash"), ("PM_Avg_Real_2010", "Average, real (2010 S$)", AQUA, "dash")]:
    fig.add_trace(go.Scatter(x=df.Year, y=df[c], name=n, mode="lines+markers", line=dict(color=col, width=2.6, dash=dash),
                             marker=dict(size=5), hovertemplate=n + " S$%{y:,.0f}<extra></extra>"))
    fig.add_annotation(x=Y1, y=df[c].iloc[-1], text=f"<b>S${df[c].iloc[-1]:,.0f}</b>", showarrow=False, xanchor="left",
                       xshift=6, yshift=-7 if "real" in n else 7, font=dict(color=col, size=11))
fig.update_yaxes(title_text="S$ per member per month", tickprefix="$", tickformat=",.0f")
fig.update_xaxes(range=[Y0 - 0.5, Y1 + 1.6])
fig.update_layout(title="Income per household member: nominal and real (resident employed households)")
add_tfr_axis(style(fig, 440)).show()

for c in ["PM_Median", "PM_Median_Real_2010", "PM_Avg", "PM_Avg_Real_2010"]:
    print(f"{c:22s} {Y0}: S${clean[c].loc[Y0]:,.0f}  {Y1}: S${clean[c].loc[Y1]:,.0f}  ({(clean[c].loc[Y1] / clean[c].loc[Y0] - 1) * 100:+.0f}%)")
print(f"TFR {Y0}: {T.loc[Y0]:.2f}  {Y1}: {T.loc[Y1]:.2f}  ({(T.loc[Y1] / T.loc[Y0] - 1) * 100:+.0f}%)")

PM_Median              2010: S$1,850  2025: S$3,909  (+111%)
PM_Median_Real_2010    2010: S$1,850  2025: S$2,909  (+57%)
PM_Avg                 2010: S$2,711  2025: S$5,173  (+91%)
PM_Avg_Real_2010       2010: S$2,711  2025: S$3,850  (+42%)
TFR 2010: 1.15  2025: 0.87  (-24%)


**Findings.**
- Median income per member rose from S\$1,850 (2010) to S\$3,909 (2025), **+111%**. After inflation it is S\$2,909 in 2010 dollars, **+57%**.
- The average rose from S\$2,711 to S\$5,173 (+91%), or S\$3,850 in 2010 dollars (+42%).
- Over the same years TFR fell from 1.15 to 0.87 (−24%). Households became richer in real terms while births fell, so a lack of income is not what lowered the birth rate.

## Chart 2 · Real income growth by income group

Source: SingStat 17885 · real % change in average income per member vs the previous year; D1–D9 = income deciles.

In [5]:
cols = ["Real_Growth_%"] + [f"Real_Growth_D{i}" for i in range(1, 10)]
z = clean[cols].dropna(how="all").T
fig = go.Figure(go.Heatmap(z=z.values, x=z.columns, y=["<b>All households</b>", "D1 lowest"] + [f"D{i}" for i in range(2, 10)],
                           colorscale=[[0, RED], [7 / 18, "#efeeea"], [1, BLUE]], zmin=-7, zmax=11, xgap=1, ygap=1,
                           text=np.round(z.values, 1), texttemplate="%{text:.1f}", textfont=dict(size=9),
                           colorbar=dict(title="%", thickness=10), hovertemplate="%{y}, %{x}: %{z:+.1f}%<extra></extra>"))
fig.add_hline(y=0.5, line=dict(color="white", width=4))
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Real income growth by income group (% vs previous year)")
style(fig, 420).show()

g = clean.loc[Y0:2024]
low, high = g[["Real_Growth_D1", "Real_Growth_D2", "Real_Growth_D3"]].mean(axis=1), g[["Real_Growth_D7", "Real_Growth_D8", "Real_Growth_D9"]].mean(axis=1)
print("years when every group fell:", [y for y in g.index if (g.loc[y, cols] < 0).all()])
print("years when all households fell:", g.index[g["Real_Growth_%"] < 0].tolist())
print(f"average yearly real growth: D1-D3 {low.mean():.1f}%, D7-D9 {high.mean():.1f}%; D1-D3 grew faster in {(low > high).sum()} of {len(g)} years")

years when every group fell: [2020]
years when all households fell: [2013, 2020]
average yearly real growth: D1-D3 3.2%, D7-D9 2.6%; D1-D3 grew faster in 8 of 15 years


**Findings.**
- Real income grew in almost every year for almost every group. All households together fell only in 2013 and 2020; **2020 is the only year every group fell**.
- Lower-income groups grew a little faster: D1–D3 averaged 3.2% a year against 2.6% for D7–D9 (2010–2024), faster in 8 of 15 years — consistent with the falling Gini in Chart 3.
- Rising real income across the whole distribution did not stop TFR falling.

## Chart 3 · Gini coefficient before and after transfers & taxes, with TFR on top

Source: SingStat 17892, M810091 · 0 = everyone has the same income, 1 = one household has everything.

In [6]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.3, 0.7], vertical_spacing=0.06)
fig.add_trace(go.Scatter(x=df.Year, y=df.TFR, name="TFR", mode="lines+markers", line=dict(color=BLUE, width=2.4),
                         marker=dict(size=5, symbol="diamond")), row=1, col=1)
for c, n, col in [("Gini_Before_Transfers", "Before transfers & taxes", VIOLET), ("Gini_After_Transfers", "After transfers & taxes", GOLD)]:
    fig.add_trace(go.Scatter(x=df.Year, y=df[c], name=n, mode="lines+markers", line=dict(color=col, width=3), marker=dict(size=6),
                             hovertemplate=n + " %{y:.3f}<extra></extra>"), row=2, col=1)
    for y, anc in [(Y0, "right"), (Y1, "left")]:
        fig.add_annotation(x=y, y=clean.loc[y, c], text=f"<b>{clean.loc[y, c]:.3f}</b>", showarrow=False, xanchor=anc,
                           xshift=-7 if anc == "right" else 7, font=dict(color=col), row=2, col=1)
fig.update_xaxes(range=[Y0 - 1.6, Y1 + 1.6])
fig.update_yaxes(title_text="TFR", row=1, col=1); fig.update_yaxes(title_text="Gini", tickformat=".2f", row=2, col=1)
fig.update_layout(title="Income inequality (Gini) before and after transfers & taxes, with TFR")
style(fig, 500).show()
print(clean[["Gini_Before_Transfers", "Gini_After_Transfers"]].loc[[2010, 2012, 2019, 2020, 2025]])

      Gini_Before_Transfers  Gini_After_Transfers
Year                                             
2010                  0.472                 0.425
2012                  0.478                 0.432
2019                  0.452                 0.398
2020                  0.452                 0.375
2025                  0.426                 0.359


**Findings.**
- **Before transfers & taxes** the Gini was 0.472 in 2010, peaked at 0.478 in 2012, then eased down to 0.426 in 2025.
- **After transfers & taxes** it was 0.425 in 2010, peaked at 0.432 in 2012, fell gradually to 0.398 in 2019, dropped to 0.375 in 2020 and kept falling to 0.359 in 2025.
- Inequality fell while TFR fell, so in levels the two are strongly linked (after transfers r = +0.89, p < 0.001) — but both simply trend down over time, so the yearly test below matters more.

## Chart 4 · Gini vs TFR: r and p (with the yearly finding)

Source: SingStat 17892, M810361, M810091 · Yearly Δ, lag L: r of ΔGini[t] and ΔTFR[t+L] · ΔGini[t] = Gini[t] − Gini[t−1].

In [7]:
COMP = [("Levels", 0, "Levels"), ("Yearly change", 0, "Yearly Δ")] + [("Yearly change", L, f"Yearly Δ, lag {L}") for L in (1, 2, 3)]
gini_table = pd.DataFrame({name: [f"{r:+.2f} ({p_text(p)})" + (" *" if p < 0.05 else "") for r, p, n in
                                  [r_of(c, m, L) for m, L, _ in COMP]]
                           for c, name in [("Gini_Before_Transfers", "Gini before"), ("Gini_After_Transfers", "Gini after")]},
                          index=[lab for *_, lab in COMP])
gini_table.index.name = "r (p) vs TFR   (* p < 0.05)"
display(gini_table)

dT = T.diff()
for lab, x in [("top 10% real income growth", clean.Real_Growth_D10_M810361), ("lowest 10% real income growth", clean.Real_Growth_D1_M810361)]:
    d = pd.concat([x, dT], axis=1).dropna(); r, p = pearsonr(d.iloc[:, 0], d.iloc[:, 1])
    print(f"{lab} vs ΔTFR: r {r:+.2f}, p {p_text(p)}, n {len(d)}")

,Gini before,Gini after
r (p) vs TFR (* p < 0.05),,
Levels,+0.91 (<0.001) *,+0.89 (<0.001) *
Yearly Δ,+0.70 (0.003) *,+0.65 (0.009) *
"Yearly Δ, lag 1",-0.21 (0.47),-0.48 (0.08)
"Yearly Δ, lag 2",+0.02 (0.94),+0.19 (0.54)
"Yearly Δ, lag 3",+0.29 (0.36),+0.41 (0.19)


top 10% real income growth vs ΔTFR: r +0.67, p 0.007, n 15
lowest 10% real income growth vs ΔTFR: r -0.06, p 0.84, n 15


**Finding.** Gini and TFR are positively linked: Gini higher, TFR higher. In levels this is two downward trends (after transfers r = +0.89, p < 0.001) — income became more equal while births fell. Unlike income, the Gini also passes the yearly test: in years when inequality rose, TFR rose too (**r = +0.65, p = 0.009, n = 15**; before transfers r = +0.70, p = 0.003). With a 1–3-year lag it disappears (lag 1: r = −0.48, p = 0.08).

**Possible explanation.** Inequality rises in good years, when top incomes (bonuses, investment income) grow most. Real income growth of the **top 10%** tracks the change in TFR (r = +0.67, p = 0.007), while the **lowest 10%** does not (r = −0.06, p = 0.84). Good years may lift both top pay and births; in years when top incomes fell (2013, 2020), births fell too. So the economic cycle, not inequality itself, likely moves births — and because the link vanishes with a lag, it is not a policy lever.

## Chart 5 · Income vs TFR: r and p for every comparison

Source: SingStat 17872, M213751, M810091 · Levels: r of x[t] and TFR[t] · Yearly Δ, lag L: r of Δx[t] and ΔTFR[t+L] · bold = p < 0.05.

In [8]:
SER = {"PM_Median": "Median income / member", "PM_Median_Real_2010": "Real median income / member",
       "PM_Avg": "Average income / member", "PM_Avg_Real_2010": "Real average income / member"}
rows, R, TXT = [], [], []
for c, name in SER.items():
    rr, tt = [], []
    for m, L, lab in COMP:
        r, p, n = r_of(c, m, L)
        rows.append(dict(series=name, comparison=lab, r=round(r, 2), p=round(p, 4), n=n, significant=p < 0.05))
        rr.append(r); t = f"r {r:+.2f} · p {p_text(p)} · n {n}"; tt.append(f"<b>{t}</b>" if p < 0.05 else t)
    R.append(rr); TXT.append(tt)
fig = go.Figure(go.Heatmap(z=R, x=[c[2] for c in COMP], y=list(SER.values()), zmin=-1, zmax=1, colorscale=DIVERGING,
                           xgap=3, ygap=3, text=TXT, texttemplate="%{text}", colorbar=dict(thickness=10)))
fig.update_xaxes(side="top"); fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Income vs TFR: Pearson r and p, 2010–2025")
style(fig, 360).update_layout(margin=dict(t=100)).show()
stats_table = pd.DataFrame(rows)
stats_table

,series,comparison,r,p,n,significant
0,Median income / member,Levels,-0.88,0.0000,16,True
1,Median income / member,Yearly Δ,-0.29,0.2909,15,False
2,Median income / member,"Yearly Δ, lag 1",-0.20,0.4863,14,False
3,Median income / member,"Yearly Δ, lag 2",0.11,0.7297,13,False
4,Median income / member,"Yearly Δ, lag 3",-0.09,0.7808,12,False
5,Real median income / member,Levels,-0.83,0.0001,16,True
6,Real median income / member,Yearly Δ,-0.23,0.4127,15,False
7,Real median income / member,"Yearly Δ, lag 1",-0.08,0.7800,14,False
8,Real median income / member,"Yearly Δ, lag 2",0.31,0.3073,13,False
9,Real median income / member,"Yearly Δ, lag 3",0.07,0.8321,12,False


**Findings.**
- **Levels:** every income series is strongly and significantly linked to TFR (median nominal r = −0.88, real r = −0.83, both p < 0.001) — but that is two trends moving in opposite directions, not evidence that income lowers births.
- **Yearly change, with or without a 1–3-year lag:** no income series is significant (every p > 0.2; e.g. real median r = −0.23, p = 0.41; lag 1 r = −0.08, p = 0.78). Nominal or real makes no difference.

## Chart 6 · Income vs TFR scatterplots: every series, comparison and lag

These are all the scatterplots the dashboard's **Income vs TFR** chart can show: 6 series × 2 comparisons (Levels, Yearly change) × 4 lags (same year, TFR 1–3 years later).
In each figure, the top row is Levels: r of x[t] and TFR[t+L]. The bottom row is Yearly change: r of Δx[t] = x[t] − x[t−1] and ΔTFR[t+L].
Each dot is one year, and darker dots are more recent years; the dashed line is the least-squares fit. Source: SingStat 17872, M213751, 17892, M810091.

In [9]:
LINK = {"PM_Median": "Median income / member", "PM_Median_Real_2010": "Real median income / member",
        "PM_Avg": "Average income / member", "PM_Avg_Real_2010": "Real average income / member",
        "Gini_Before_Transfers": "Gini before transfers", "Gini_After_Transfers": "Gini after transfers"}
MODES = ["Levels", "Yearly change"]


def link_scatters(x):
    """Same scatter as the dashboard's Income vs TFR chart, for every comparison and lag (2 rows x 4 columns)."""
    name, panels, titles = LINK[x], [], []
    for mode in MODES:
        for L in range(4):
            d = pair(x, mode, L)
            r, p = pearsonr(d.x, d.y)
            lag = "same year" if L == 0 else f"TFR {L} yr later"
            titles.append(f"{'Levels' if mode == 'Levels' else 'Yearly Δ'}, {lag}<br><sup>r {r:+.2f}, p {p_text(p)}, n {len(d)} "
                          f"({'significant' if p < 0.05 else 'not significant'})</sup>")
            panels.append((mode, L, d))
    fig = make_subplots(rows=2, cols=4, subplot_titles=titles, horizontal_spacing=0.06, vertical_spacing=0.22)
    for k, (mode, L, d) in enumerate(panels):
        row, col = k // 4 + 1, k % 4 + 1
        yrs = d.index.values.astype(int)
        if mode != "Levels":
            fig.add_hline(y=0, line=dict(color=LIGHT, width=1), row=row, col=col)
            fig.add_vline(x=0, line=dict(color=LIGHT, width=1), row=row, col=col)
        b, a = np.polyfit(d.x, d.y, 1)
        xs = np.linspace(d.x.min(), d.x.max(), 30)
        fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color=MUTED, dash="dash", width=1.4),
                                 hoverinfo="skip"), row=row, col=col)
        ends = {yrs.min(), yrs.max()}
        fig.add_trace(go.Scatter(
            x=d.x, y=d.y, mode="markers+text", text=[str(y) if y in ends else "" for y in yrs], textposition="top center",
            textfont=dict(size=9, color=MUTED), customdata=yrs,
            marker=dict(size=10, color=yrs, colorscale="Blues", cmin=yrs.min() - 0.5 * (yrs.max() - yrs.min() + 1),
                        cmax=yrs.max(), line=dict(color="white", width=1.2)),
            hovertemplate="<b>%{customdata}</b><br>" + name + ": %{x:,.3f}<br>TFR: %{y:,.3f}<extra></extra>"), row=row, col=col)
        fig.update_xaxes(title_text=name if mode == "Levels" else "change vs previous year", showgrid=True, gridcolor=GRID,
                         row=row, col=col)
    fig.update_yaxes(title_text="TFR", row=1, col=1)
    fig.update_yaxes(title_text="change in TFR", row=2, col=1)
    style(fig, 760)
    fig.update_annotations(font_size=12)
    fig.update_layout(title=f"{name} vs TFR", showlegend=False, hovermode="closest", margin=dict(l=10, r=10, t=110, b=10))
    fig.show()

In [10]:
link_scatters("PM_Median")

**Median income per member (nominal).** Levels: strongly negative at every lag (r −0.88 to −0.94, all p < 0.001), because the two series trend in opposite directions. Yearly change: no link at any lag (same year r −0.29, p 0.29; lag 1 −0.20, p 0.49; lag 2 +0.11, p 0.73; lag 3 −0.09, p 0.78).

In [11]:
link_scatters("PM_Median_Real_2010")

**Real median income per member (2010 S$).** Levels: r −0.83 to −0.91, all p < 0.001. Yearly change: same year r −0.23, p 0.41; lag 1 −0.08, p 0.78; lag 2 +0.31, p 0.31; lag 3 +0.07, p 0.83. None of these is significant.

In [12]:
link_scatters("PM_Avg")

**Average income per member (nominal).** Levels: r −0.86 to −0.93, all p < 0.001. Yearly change: same year r +0.12, p 0.68; lag 1 −0.34, p 0.24; lag 2 +0.17, p 0.57; lag 3 +0.06, p 0.85. None of these is significant.

In [13]:
link_scatters("PM_Avg_Real_2010")

**Real average income per member (2010 S$).** Levels: r −0.76 to −0.86, all p < 0.001. Yearly change: same year r +0.25, p 0.37; lag 1 −0.25, p 0.38; lag 2 +0.29, p 0.34; lag 3 +0.20, p 0.52. None of these is significant.

In [14]:
link_scatters("Gini_Before_Transfers")

**Gini before transfers & taxes.** Levels: r +0.91 to +0.95, all p < 0.001 (both series trend down). Yearly change: **same year r +0.70, p 0.003 (significant)**. With a lag, the link is not significant: lag 1 −0.21, p 0.47; lag 2 +0.02, p 0.94; lag 3 +0.29, p 0.36.

In [15]:
link_scatters("Gini_After_Transfers")

**Gini after transfers & taxes.** Levels: r +0.87 to +0.93, all p < 0.001. Yearly change: **same year r +0.65, p 0.009 (significant)**. With a lag, the link is not significant: lag 1 −0.48, p 0.08; lag 2 +0.19, p 0.54; lag 3 +0.41, p 0.19.

In [16]:
rows = []
for x, name in LINK.items():
    for mode in MODES:
        for L in range(4):
            d = pair(x, mode, L); r, p = pearsonr(d.x, d.y)
            rows.append(dict(series=name, comparison=mode, lag=L, r=round(r, 2), p=round(p, 4), n=len(d), significant=p < 0.05))
all_scatter_stats = pd.DataFrame(rows)
all_scatter_stats

,series,comparison,lag,r,p,n,significant
0,Median income / member,Levels,0,-0.88,0.0000,16,True
1,Median income / member,Levels,1,-0.93,0.0000,15,True
2,Median income / member,Levels,2,-0.94,0.0000,14,True
3,Median income / member,Levels,3,-0.92,0.0000,13,True
4,Median income / member,Yearly change,0,-0.29,0.2909,15,False
5,Median income / member,Yearly change,1,-0.20,0.4863,14,False
6,Median income / member,Yearly change,2,0.11,0.7297,13,False
7,Median income / member,Yearly change,3,-0.09,0.7808,12,False
8,Real median income / member,Levels,0,-0.83,0.0001,16,True
9,Real median income / member,Levels,1,-0.87,0.0000,15,True


**Findings across all 48 scatterplots.**
- **Levels:** every series is significant at every lag. This is the shared long-run trend, not evidence of a link: income rose while TFR fell, and the Gini fell together with TFR.
- **Yearly change:** none of the four income series is significant at any lag; every p-value is above 0.2.
- **Gini, yearly change:** the only significant result is in the same year (before transfers r +0.70, p 0.003; after transfers r +0.65, p 0.009). It disappears with a 1–3-year lag.

## Summary

1. Income per member rose strongly, even after inflation (median +57% real), while TFR fell 24%.
2. Income's strong correlation with TFR exists only in levels (shared trends); yearly and lagged tests are all insignificant (p > 0.2).
3. Inequality fell (Gini after 0.425 → 0.359). Higher Gini goes with higher TFR year to year (p = 0.009), possibly because top incomes and births both follow the economy; it disappears with a lag, so it is not a policy lever.
4. Conclusion: money is not the main brake on births in Singapore. 